In [1]:
import digitalhub as dh

project = dh.get_or_create_project("nori-tvm-xinet")

In [18]:
model = dh.log_model(
    project="nori-tvm-xinet",
    name="xinet-pose-224",
    source="./XiNet-s-pose-224.onnx",
    framework="onnx"
)

In [19]:
xinet_tvm_func = project.new_function(name="xinet-pose-tvm", kind="tvm",
                                  model="store://nori-tvm-xinet/model/model/xinet-pose-224", 
                                  format="auto")

In [ ]:
xinet_tvm_build = xinet_tvm_func.run(
    action="build",
    resources={"cpu":"1", "mem":"2Gi"},
    wait=False
)

In [ ]:
xinet_tvm_build = xinet_tvm_func.run(
    action="compile",
    profile="jetson-orin",
    target_architecture="arm64",
    tag="arm",
    opt_level=3,
    resources={"cpu":"1", "mem":"4Gi"},
    wait=False
)

In [ ]:
xinet_tvm_func.run(
    action="serve", 
    profile="jetson-orin",
    served_name="xinet-pose-224",
    resources={"cpu":"1", "mem":"1Gi"}, 
    wait=False)

In [2]:
func = project.new_function(name="xinet-pose",
                            kind="openinference",
                            python_version="PYTHON3_13",
                            code_src="src",
                            handler="xinet_tvm_handler:handler",
                            init_function="init_context",
                            model_name="XiNet-s-pose-224",
                            inputs=[{
                                "name": "image",
                                "datatype": "UINT8",
                                "shape": [-1,-1]
                            }],
                            outputs=[{
                                "name": "image",
                                "datatype": "UINT8",
                                "shape": [-1,-1]
                            }],
                            requirements=["opencv-python==4.13.0.92", "numpy==2.4.3", "onnxruntime==1.24.3", "matplotlib==3.10.8"]
                           )

In [3]:
build = func.run(
    "build",
    instructions=[
        "apt-get update && apt-get install -y libgl1 libglib2.0-0 libsm6 libxext6 libxrender1 libfontconfig1 libice6"
    ],
    resources={"mem":"1Gi"},
    wait=False
)

In [4]:
run = func.run(
    action="serve", 
    resources={"mem": "1Gi", "disk": "1Gi"}, 
    init_parameters={"tvm_func":"http://s-nori-tvm-xinet-xinet-pose-tvm-latest.digitalhub:8080/v2/models/xinet-pose-224"})

In [5]:
stream_func = project.new_function("videostream", kind="container", image="alexxit/go2rtc", command="/bin/bash", code_src="go2rtc")

In [ ]:
stream_func.run(action="serve", 
                service_ports=[{"port": 1984, "target_port": 1984}], 
                fs_group=8877, run_as_user=8877, run_as_group=8877, 
                args=["/shared/lunch_go2rtc.sh"],
                resources={"mem":"512Mi"},
                wait=False)

In [2]:
graph_func = project.new_function("xinet-pipeline", kind="servicegraph", code_src="pipeline.yaml")

In [4]:
graph_run = graph_func.run(action="serve", 
    parameters={
    "input.url": "http://s-nori-tvm-xinet-videostream-latest.digitalhub:1984/api/stream.mjpeg?src=example",
    "xinet-pose-service.address": "s-nori-tvm-xinet-xinet-pose-latest.digitalhub:9000"}, 
    service_ports=[{"port": 7777, "target_port": 7777}],
    resources={"mem":"512Mi"})